# Optional  Lab: Cost Function 
<figure>
    <center> <img src="./images/C1_W1_L3_S2_Lecture_b.png"  style="width:1000px;height:200px;" ></center>
</figure>



## Goals
In this lab you will:
- you will implement and explore the `cost` function for linear regression with one variable. 


## Tools
In this lab we will make use of: 
- NumPy, a popular library for scientific computing
- Matplotlib, a popular library for plotting data
- local plotting routines in the lab_utils_uni.py file in the local directory

In [2]:
# Importações necessárias
import numpy as np
import matplotlib.pyplot as plt

# Importações específicas da biblioteca do exercício
from lab_utils_uni import plt_intuition, plt_stationary, plt_update_onclick, soup_bowl

# Função para configurar o ambiente de plotagem
def configure_plotting_style(style_path: str = './deeplearning.mplstyle') -> None:
    """
    Configura o estilo para os gráficos do Matplotlib.
    Args:
        style_path: Caminho para o arquivo de estilo (.mplstyle).
    """
    try:
        plt.style.use(style_path)
        print(f"Estilo de plotagem configurado: {style_path}")
    except FileNotFoundError:
        print("Arquivo de estilo não encontrado. Usando estilo padrão.")
        plt.style.use('default')

# Configuração do estilo de plotagem
configure_plotting_style()

# Ativando widgets interativos do Matplotlib
%matplotlib widget

Estilo de plotagem configurado: ./deeplearning.mplstyle


## Problem Statement

You would like a model which can predict housing prices given the size of the house.  
Let's use the same two data points as before the previous lab- a house with 1000 square feet sold for \\$300,000 and a house with 2000 square feet sold for \\$500,000.


| Size (1000 sqft)     | Price (1000s of dollars) |
| -------------------| ------------------------ |
| 1                 | 300                      |
| 2                  | 500                      |


In [3]:
# Dados de entrada armazenados de forma clara e expansível
training_data = {
    "size_in_1000sqft": [1.0, 2.0],  # Tamanhos das casas (em 1000 pés quadrados)
    "price_in_1000usd": [300.0, 500.0]  # Preços das casas (em 1000 dólares)
}

# Conversão para arrays NumPy
x_train = np.array(training_data["size_in_1000sqft"], dtype=float)
y_train = np.array(training_data["price_in_1000usd"], dtype=float)

# Validação para garantir que os tamanhos de x_train e y_train sejam consistentes
assert len(x_train) == len(y_train), "Os tamanhos de x_train e y_train não correspondem!"

# Exibição dos dados para validação
print("Tamanho dos dados de treino:")
print(f"x_train (tamanho): {x_train}")
print(f"y_train (preço): {y_train}")

Tamanho dos dados de treino:
x_train (tamanho): [1. 2.]
y_train (preço): [300. 500.]


## Computing Cost
The term 'cost' in this assignment might be a little confusing since the data is housing cost. Here, cost is a measure how well our model is predicting the target price of the house. The term 'price' is used for housing data.

The equation for cost with one variable is:
  $$J(w,b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})^2 \tag{1}$$ 
 
where 
  $$f_{w,b}(x^{(i)}) = wx^{(i)} + b \tag{2}$$
  
- $f_{w,b}(x^{(i)})$ is our prediction for example $i$ using parameters $w,b$.  
- $(f_{w,b}(x^{(i)}) -y^{(i)})^2$ is the squared difference between the target value and the prediction.   
- These differences are summed over all the $m$ examples and divided by `2m` to produce the cost, $J(w,b)$.  
>Note, in lecture summation ranges are typically from 1 to m, while code will be from 0 to m-1.


The code below calculates cost by looping over each example. In each loop:
- `f_wb`, a prediction is calculated
- the difference between the target and the prediction is calculated and squared.
- this is added to the total cost.

In [4]:
def compute_cost(x, y, w, b):
    """
    Computes the cost function for linear regression using vectorized operations.
    
    Args:
        x (ndarray (m,)): Input feature data (e.g., size in 1000 square feet), m examples.
        y (ndarray (m,)): Target values (e.g., price in 1000s of dollars).
        w (float): Weight parameter for the model.
        b (float): Bias parameter for the model.
    
    Returns:
        total_cost (float): The cost of using w, b as the parameters for linear regression
                            to fit the data points in x and y.
    
    Raises:
        ValueError: If the size of x and y do not match.
    
    Example:
        >>> x = np.array([1.0, 2.0])
        >>> y = np.array([300.0, 500.0])
        >>> compute_cost(x, y, w=100, b=50)
        0.0
    """
    # Verificar se o número de exemplos em x e y é consistente
    if x.shape[0] != y.shape[0]:
        raise ValueError("x e y devem ter o mesmo número de exemplos.")
    
    # Número de exemplos de treino
    m = x.shape[0]
    
    # Previsões do modelo
    f_wb = w * x + b  # Operação vetorizada para calcular f_wb
    
    # Diferenças ao quadrado
    errors_squared = (f_wb - y) ** 2  # Vetor com os erros ao quadrado
    
    # Cálculo do custo total
    total_cost = (1 / (2 * m)) * np.sum(errors_squared)
    
    return total_cost

## Cost Function Intuition

<img align="left" src="./images/C1_W1_Lab02_GoalOfRegression.PNG"    style=" width:380px; padding: 10px;  " /> Your goal is to find a model $f_{w,b}(x) = wx + b$, with parameters $w,b$,  which will accurately predict house values given an input $x$. The cost is a measure of how accurate the model is on the training data.

The cost equation (1) above shows that if $w$ and $b$ can be selected such that the predictions $f_{w,b}(x)$ match the target data $y$, the $(f_{w,b}(x^{(i)}) - y^{(i)})^2 $ term will be zero and the cost minimized. In this simple two point example, you can achieve this!

In the previous lab, you determined that $b=100$ provided an optimal solution so let's set $b$ to 100 and focus on $w$.

<br/>
Below, use the slider control to select the value of $w$ that minimizes cost. It can take a few seconds for the plot to update.

In [7]:
# Visualize a intuição da função de custo para a regressão linear
from lab_utils_uni import plt_intuition  # Importando a função necessária do módulo lab_utils_uni

# Plotar a intuição da função de custo
plt_intuition(x_train, y_train)

# Notas importantes:
# - Esse gráfico ajuda a visualizar como os valores de w (peso) e b (viés) afetam o custo.
# - Experimente diferentes valores de w no controle deslizante para entender a relação.
# - O custo é minimizado quando w = 200, conforme observado em experimentos anteriores.

interactive(children=(IntSlider(value=150, description='w', max=400, step=10), Output()), _dom_classes=('widge…

The plot contains a few points that are worth mentioning.
- cost is minimized when $w = 200$, which matches results from the previous lab
- Because the difference between the target and pediction is squared in the cost equation, the cost increases rapidly when $w$ is either too large or too small.
- Using the `w` and `b` selected by minimizing cost results in a line which is a perfect fit to the data.

## Cost Function Visualization- 3D

You can see how cost varies with respect to *both* `w` and `b` by plotting in 3D or using a contour plot.   
It is worth noting that some of the plotting in this course can become quite involved. The plotting routines are provided and while it can be instructive to read through the code to become familiar with the methods, it is not needed to complete the course successfully. The routines are in lab_utils_uni.py in the local directory.

### Larger Data Set
It is instructive to view a scenario with a few more data points. This data set includes data points that do not fall on the same line. What does that mean for the cost equation? Can we find $w$, and $b$ that will give us a cost of 0? 

In [8]:
import numpy as np

# Define o conjunto de dados de treinamento
x_train = np.array([1.0, 1.7, 2.0, 2.5, 3.0, 3.2])  # Variável independente (eixo x)
y_train = np.array([250, 300, 480, 430, 630, 730])  # Variável dependente (eixo y)

# Garantia de que os arrays têm o mesmo comprimento
assert len(x_train) == len(y_train), "x_train e y_train devem ter o mesmo número de elementos"

In the contour plot, click on a point to select `w` and `b` to achieve the lowest cost. Use the contours to guide your selections. Note, it can take a few seconds to update the graph. 

In [9]:
# Fecha todas as figuras abertas para evitar sobreposição de gráficos
plt.close('all') 

# Criação da figura e dos eixos para visualização dos dados
fig, ax, dyn_items = plt_stationary(x_train, y_train)

# Configuração do atualizador para interação com o gráfico
updater = plt_update_onclick(fig, ax, x_train, y_train, dyn_items)

# Adicionando comentários e boas práticas para clareza
# - `plt_stationary`: Cria o gráfico inicial com os dados de x_train e y_train.
# - `plt_update_onclick`: Habilita a interação no gráfico ao clicar em pontos.

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Above, note the dashed lines in the left plot. These represent the portion of the cost contributed by each example in your training set. In this case, values of approximately $w=209$ and $b=2.4$ provide low cost. Note that, because our training examples are not on a line, the minimum cost is not zero.

### Convex Cost surface
The fact that the cost function squares the loss ensures that the 'error surface' is convex like a soup bowl. It will always have a minimum that can be reached by following the gradient in all dimensions. In the previous plot, because the $w$ and $b$ dimensions scale differently, this is not easy to recognize. The following plot, where $w$ and $b$ are symmetric, was shown in lecture:

In [10]:
import numpy as np
import matplotlib.pyplot as plt

def soup_bowl():
    # Criação de uma grade de valores para w e b
    w = np.linspace(-10, 10, 100)  # Intervalo para w
    b = np.linspace(-10, 10, 100)  # Intervalo para b
    W, B = np.meshgrid(w, b)  # Criação de uma grade para cálculo

    # Definição da função de custo convexa (exemplo: erro quadrático)
    Z = W**2 + B**2

    # Configuração do gráfico
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    # Plot da superfície convexa
    surf = ax.plot_surface(W, B, Z, cmap='viridis', edgecolor='k', alpha=0.8)
    ax.set_title("Convex Cost Surface (Soup Bowl)")
    ax.set_xlabel('w')
    ax.set_ylabel('b')
    ax.set_zlabel('Cost')

    # Adicionar barra de cores
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10)

    # Exibição do gráfico
    plt.show()

# Chamar a função para visualizar o gráfico
soup_bowl()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

# Congratulations!
You have learned the following:
 - The cost equation provides a measure of how well your predictions match your training data.
 - Minimizing the cost can provide optimal values of $w$, $b$.

In [11]:
import numpy as np
import matplotlib.pyplot as plt

# Função para criar o gráfico da superfície convexa
def soup_bowl():
    # Criação de uma grade de valores para w e b
    w = np.linspace(-10, 10, 100)  # Intervalo para w
    b = np.linspace(-10, 10, 100)  # Intervalo para b
    W, B = np.meshgrid(w, b)  # Criação de uma grade para cálculo

    # Definição da função de custo convexa (exemplo: erro quadrático)
    Z = W**2 + B**2

    # Configuração do gráfico
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    # Plot da superfície convexa
    surf = ax.plot_surface(W, B, Z, cmap='viridis', edgecolor='k', alpha=0.8)
    ax.set_title("Convex Cost Surface (Soup Bowl)")
    ax.set_xlabel('w')
    ax.set_ylabel('b')
    ax.set_zlabel('Cost')

    # Adicionar barra de cores
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10)

    # Exibição do gráfico
    plt.show()

# Dados de treinamento
x_train = np.array([1.0, 1.7, 2.0, 2.5, 3.0, 3.2])  # Variável independente (eixo x)
y_train = np.array([250, 300, 480, 430, 630, 730])  # Variável dependente (eixo y)

# Garantia de que os arrays têm o mesmo comprimento
assert len(x_train) == len(y_train), "x_train e y_train devem ter o mesmo número de elementos"

# Funções fictícias (substitua por suas implementações reais, se necessário)
def plt_stationary(x_train, y_train):
    """
    Cria o gráfico inicial para os dados de treinamento.
    """
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(x_train, y_train, label="Training Data", color="blue", s=50)
    ax.set_title("Training Data Visualization")
    ax.set_xlabel("x_train")
    ax.set_ylabel("y_train")
    ax.legend()
    
    # Elementos dinâmicos fictícios
    dyn_items = {"example_item": "placeholder"}
    return fig, ax, dyn_items

def plt_update_onclick(fig, ax, x_train, y_train, dyn_items):
    """
    Habilita a interação ao clicar no gráfico.
    """
    def onclick(event):
        # Exemplo de interação: exibe coordenadas do clique
        print(f"Clicked at x={event.xdata}, y={event.ydata}")

    # Conecta o evento de clique ao gráfico
    cid = fig.canvas.mpl_connect('button_press_event', onclick)
    return cid

# Passos de visualização
plt.close('all')  # Fecha gráficos anteriores
fig, ax, dyn_items = plt_stationary(x_train, y_train)  # Cria gráfico inicial
updater = plt_update_onclick(fig, ax, x_train, y_train, dyn_items)  # Habilita interação

# Exibição da superfície convexa
soup_bowl()  # Exibe o gráfico da superfície convexa

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [12]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # Necessário para plotagens 3D

def plot_convex_cost_surface():
    """
    Plota a superfície de custo convexa ("Soup Bowl") usando a função:
        Cost = w^2 + b^2
    Essa função cria uma grade de valores para w e b e exibe a superfície 3D resultante.
    """
    # Definir intervalos para w e b
    w_range = np.linspace(-10, 10, 100)  # Valores para w
    b_range = np.linspace(-10, 10, 100)  # Valores para b
    W, B = np.meshgrid(w_range, b_range)  # Cria uma grade 2D a partir dos intervalos

    # Função de custo convexa (exemplo: erro quadrático simples)
    Z = W**2 + B**2

    # Configuração do gráfico 3D
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    surface = ax.plot_surface(W, B, Z, cmap='viridis', edgecolor='k', alpha=0.8)
    
    # Configurar títulos e rótulos dos eixos
    ax.set_title("Superfície de Custo Convexa (Soup Bowl)")
    ax.set_xlabel('w')
    ax.set_ylabel('b')
    ax.set_zlabel('Custo')
    
    # Adicionar uma barra de cores para indicar os níveis de custo
    fig.colorbar(surface, ax=ax, shrink=0.5, aspect=10)
    
    plt.show()

def plot_training_data(x_train, y_train):
    """
    Plota os dados de treinamento em um gráfico de dispersão.
    
    Parâmetros:
        x_train (np.array): Valores da variável independente.
        y_train (np.array): Valores da variável dependente.
    
    Retorna:
        fig: objeto figura do matplotlib.
        ax: objeto eixos do matplotlib.
    """
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(x_train, y_train, color='blue', s=50, label='Dados de Treinamento')
    ax.set_title("Visualização dos Dados de Treinamento")
    ax.set_xlabel("x_train")
    ax.set_ylabel("y_train")
    ax.legend()
    
    return fig, ax

def enable_click_interaction(fig):
    """
    Habilita a interação via clique no gráfico. Ao clicar sobre o gráfico, 
    as coordenadas do clique (x e y) são impressas no console.
    
    Parâmetros:
        fig: objeto figura do matplotlib.
        
    Retorna:
        cid: ID da conexão do evento (útil se precisar desconectar o evento posteriormente).
    """
    def on_click(event):
        # Verifica se o clique ocorreu dentro dos eixos
        if event.inaxes is not None:
            print(f"Clique detectado em x={event.xdata:.2f}, y={event.ydata:.2f}")
    
    cid = fig.canvas.mpl_connect('button_press_event', on_click)
    return cid

# Dados de treinamento fornecidos
x_train = np.array([1.0, 1.7, 2.0, 2.5, 3.0, 3.2])
y_train = np.array([250, 300, 480, 430, 630, 730])

# Garante que os arrays tenham o mesmo tamanho
assert len(x_train) == len(y_train), "x_train e y_train devem ter o mesmo número de elementos."

# Fechar gráficos anteriores
plt.close('all')

# Plotar os dados de treinamento e habilitar interação via clique
fig_train, ax_train = plot_training_data(x_train, y_train)
enable_click_interaction(fig_train)

# Exibir a superfície de custo convexa ("Soup Bowl")
plot_convex_cost_surface()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …